In [1]:
# local
from var_check import name_check
from var_check import time_check
from var_check import url_check
from sqlSrcipts import sql_connect
from sqlSrcipts import sql_engine 
# standard
import os
# third
import pandas as pd
import matplotlib.pyplot
import matplotlib.dates as mdates
import requests 

In [17]:
name = 'Gold Coin Chest'

start = "2026-01-16" #UTC Time
end = "2026-01-20"

condense = 'true'
has_sold = 'true'

limit = '50'
page = '1'

# name_check(name)
# time_check(start, end)

def url_page():
    url = (
        f'https://api.darkerdb.com/v1/market'
        f'?key={os.getenv("dark_api_key")}'
        f'&item={name}'
        f'&limit={limit}&page={page}'
        f'&condense={condense}&has_sold={has_sold}'
        f'&from={start}&to={end}')
    return url


conn = sql_connect()
cursor = conn.cursor()
query = """
    SELECT MAX(cursor) AS latest_cursor FROM test
    """
cursor.execute(query)
latest_cursor = cursor.fetchone()[0]
conn.close()

def url_cursor():
    url = (
        f'https://api.darkerdb.com/v1/market'
        f'?key={os.getenv("dark_api_key")}'
        f'&item={name}'
        f'&limit={limit}&page={page}'
        f'&condense={condense}&has_sold={has_sold}'
        f"&cursor={latest_cursor}")
    return url
# req = requests.get(url_cursor())

In [12]:
req = requests.get(url_cursor())

In [14]:
req.json()['body'][-1]['cursor']

3069260021

In [18]:
# request through cursor
with requests.Session() as ses:
    req_body = []
    json = {"pagination": {"count": int(limit)}}
    # byte_counter = 0    
    request_count = 0
    while json['pagination']['count'] != 0:
        req = ses.get(url_cursor())
        json = req.json()
        req_body.extend(json['body'])
        latest_cursor = json['body'][-1]['cursor'] #sometime req stop comming, rate limited?
        
        request_count+=1     
        print(f'{request_count}')
        # byte_counter += len(req.content)
    df = pd.json_normalize(req_body, sep=',')
    # print(f'get-requests = {page} \n bytes = {byte_counter}')

KeyboardInterrupt: 

In [92]:
# latest_cursor[0]
print(req.json()['body'][0]['cursor'])
req.json()
# url = url_cursor()
# 3069260021

3069266995


{'version': '1.0.7',
 'status': 'OK',
 'code': 200,
 'query_time': 0.0382,
 'query_date': '2026-01-18T14:21:20Z',
 'stage': 'production',
 'build': '0.14.125.7828',
 'patch': 105,
 'meta': {'method': 'GET',
  'request': 'https://api.darkerdb.com/v1/market?key=meowlin&item=Gold+Coin+Chest&limit=50&page=27&condense=true&has_sold=true&cursor=3060818957',
  'query': {'key': 'meowlin',
   'item': 'Gold Coin Chest',
   'limit': 50,
   'page': 27,
   'condense': 'true',
   'has_sold': 'true',
   'cursor': 3060818957},
  'params': []},
 'pagination': {'count': 50, 'limit': 50, 'cursor': 3060818957},
 'body': [{'id': 514139423252699136,
   'cursor': 3069266995,
   'item_id': 'GoldCoinChest',
   'item': 'Gold Coin Chest',
   'archetype': 'GoldCoinChest',
   'rarity': 'Unique',
   'price': 9999,
   'price_per_unit': 9999,
   'quantity': 1,
   'created_at': '2026-01-18T13:33:52Z',
   'expires_at': '2026-01-25T13:33:52Z',
   'sold_at': '2026-01-18T14:02:07Z',
   'has_sold': True,
   'has_expired': 

In [16]:
# request through pagination
with requests.Session() as ses:
    req_body = []
    json = {"pagination": {"count": int(limit)}}
    # byte_counter = 0    
    while json['pagination']['count'] != 0:
        req = ses.get(url_page())
        json = req.json()
        req_body.extend(json['body'])
        print(f'{page}')
        page = str(int(page) + 1) #sometime req stop comming, rate limited?
        # byte_counter += len(req.content)
    df = pd.json_normalize(req_body, sep=',')
    # print(f'get-requests = {page} \n bytes = {byte_counter}')

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27


In [ ]:
conn = sql_connect()
cursor = conn.cursor()


type_map = {
    "int64": "BIGINT",
    "float64": "DOUBLE PRECISION",
    "object": "TEXT",
    "datetime64[ns]": "TIMESTAMP",
    "bool" : "BOOLEAN"    
}
df = pd.DataFrame(req_body)
schema = {col: type_map[str(dtype)] for col, dtype in zip(df.columns, df.dtypes)}
columns_dtype = ", ".join(f"{col} {dtype}" for col, dtype in schema.items())
query = f'''
    CREATE TABLE IF NOT EXISTS test (
    {columns_dtype},
    PRIMARY KEY (cursor)
    );'''
cursor.execute(query)


sqlCol = ', '.join(df.columns)
sqlPlaceholder = ", ".join(["%s"] * len(df.columns))
query = f'''
    INSERT INTO test ({sqlCol}) 
    VALUES ({sqlPlaceholder})
    ON CONFLICT (cursor) DO NOTHING;
    '''
rows = [tuple(instance[c] for c in df.columns)for instance in req_body]
cursor.executemany(query,rows)


conn.commit()
conn.close()